In [1]:
# ============================================================
# Install Required Libraries
# ============================================================

!pip -q install groq
!pip -q install sentence-transformers
!pip -q install faiss-cpu
!pip -q install langchain
!pip -q install pandas
!pip -q install openpyxl
!pip -q install numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 82.4 MB/s eta 0:00:00


In [2]:
import groq
import faiss
import numpy
import pandas

print("All Libraries Installed Successfully")

All Libraries Installed Successfully


In [4]:
!pip install langchain langchain-core sentence-transformers groq


In [7]:
# ============================================================
# Import Required Libraries
# ============================================================

import os
import pickle
import numpy as np
import pandas as pd
import faiss

from sentence_transformers import SentenceTransformer
from langchain_core.documents import Document  # Fixed: Modern import path
from groq import Groq

print("All libraries imported successfully!")


All libraries imported successfully!


In [8]:
# ============================================================
# Mount Google Drive
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [9]:
# ============================================================
# Database Path
# ============================================================

BASE_PATH = "/content/drive/MyDrive/database"

print("Database Path:")
print(BASE_PATH)

Database Path:
/content/drive/MyDrive/database


In [11]:
# ============================================================
# Check Database Files
# ============================================================

print("Files Available:\n")

for file in os.listdir(BASE_PATH):
    print(file)

Files Available:

persons.xlsx
criminals.xlsx
officers.xlsx
evidence.xlsx
fir.xlsx
wanted.xlsx
cases.xlsx
charges.xlsx
arrests.xlsx
convictions.xlsx
warrants.xlsx
users.xlsx
merged_police_data.xlsx
documents.pkl
embeddings.npy
metadata.pkl
faiss.index


In [12]:
# ============================================================
# Verify Required RAG Files
# ============================================================

required_files = [
    "documents.pkl",
    "embeddings.npy",
    "metadata.pkl",
    "faiss.index"
]

print("Checking Required Files...\n")

for file in required_files:
    path = os.path.join(BASE_PATH, file)

    if os.path.exists(path):
        print(f"✅ {file} Found")
    else:
        print(f"❌ {file} Missing")

Checking Required Files...

✅ documents.pkl Found
✅ embeddings.npy Found
✅ metadata.pkl Found
✅ faiss.index Found


In [13]:
# ============================================================
# Load FAISS Index
# ============================================================

faiss_path = os.path.join(BASE_PATH, "faiss.index")

index = faiss.read_index(faiss_path)

print("✅ FAISS Index Loaded Successfully")
print("Total Indexed Documents:", index.ntotal)

✅ FAISS Index Loaded Successfully
Total Indexed Documents: 50


In [14]:
# ============================================================
# Load Documents
# ============================================================

documents_path = os.path.join(BASE_PATH, "documents.pkl")

with open(documents_path, "rb") as f:
    documents = pickle.load(f)

print("✅ Documents Loaded")
print("Total Documents:", len(documents))

✅ Documents Loaded
Total Documents: 50


In [15]:
# ============================================================
# Load Metadata
# ============================================================

metadata_path = os.path.join(BASE_PATH, "metadata.pkl")

with open(metadata_path, "rb") as f:
    metadata = pickle.load(f)

print("✅ Metadata Loaded")
print("Total Metadata Records:", len(metadata))

✅ Metadata Loaded
Total Metadata Records: 50


In [16]:
# ============================================================
# Load Embedding Model
# ============================================================

embedding_model = SentenceTransformer(
    "BAAI/bge-small-en-v1.5"
)

print("✅ Embedding Model Loaded Successfully")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedding Model Loaded Successfully


In [17]:
# ============================================================
# Verify Loaded Components
# ============================================================

print("=" * 60)

print("FAISS Index Size :", index.ntotal)

print("Documents :", len(documents))

print("Metadata :", len(metadata))

print("=" * 60)

FAISS Index Size : 50
Documents : 50
Metadata : 50


In [18]:
# ============================================================
# Test Embedding Generation
# ============================================================

query = "Vehicle theft case in Chennai"

query_embedding = embedding_model.encode(
    [query],
    normalize_embeddings=True
)

print("Embedding Shape:", query_embedding.shape)

Embedding Shape: (1, 384)


In [19]:
# ============================================================
# Test Vector Search
# ============================================================

scores, indices = index.search(
    query_embedding.astype("float32"),
    3
)

print("Scores:")
print(scores)

print("\nIndices:")
print(indices)

Scores:
[[0.81854355 0.8095599  0.80876833]]

Indices:
[[40 15 20]]


In [20]:
# ============================================================
# Show Retrieved Metadata
# ============================================================

for idx in indices[0]:

    print("=" * 70)

    print(metadata[idx])

{'fir_number': 'TNFIR20260041', 'case_number': 'CASE041', 'crime': 'Vehicle Theft', 'district': 'Chennai', 'station': 'T. Nagar Police Station', 'criminal': 'Ashok', 'officer': 'R.Kumar 41', 'rank': 'Inspector', 'status': 'Under Investigation'}
{'fir_number': 'TNFIR20260016', 'case_number': 'CASE016', 'crime': 'Vehicle Theft', 'district': 'Chennai', 'station': 'T. Nagar Police Station', 'criminal': 'Lakshmi', 'officer': 'K.Murugan 16', 'rank': 'Inspector', 'status': 'Under Investigation'}
{'fir_number': 'TNFIR20260021', 'case_number': 'CASE021', 'crime': 'Vehicle Theft', 'district': 'Chennai', 'station': 'T. Nagar Police Station', 'criminal': 'Bala', 'officer': 'R.Kumar 21', 'rank': 'Inspector', 'status': 'Under Investigation'}


In [1]:
# ============================================================
# Enter Groq API Key
# ============================================================

from getpass import getpass

GROQ_API_KEY = getpass("Enter your Groq API Key: ")

Enter your Groq API Key: ··········


In [2]:
# ============================================================
# Initialize Groq Client
# ============================================================

from groq import Groq

client = Groq(
    api_key=GROQ_API_KEY
)

print("✅ Groq Client Connected Successfully")

✅ Groq Client Connected Successfully


In [3]:
# ============================================================
# LLM Model
# ============================================================

MODEL_NAME = "llama-3.1-8b-instant"

print("Using Model:", MODEL_NAME)

Using Model: llama-3.1-8b-instant


In [4]:
# ============================================================
# Test LLM
# ============================================================

response = client.chat.completions.create(

    model=MODEL_NAME,

    messages=[
        {
            "role": "user",
            "content": "Say Hello from Groq."
        }
    ],

    temperature=0.2,

    max_tokens=50

)

print(response.choices[0].message.content)

Hello from Groq.


In [5]:
# ============================================================
# LLM Response Function
# ============================================================

def ask_llm(prompt):

    response = client.chat.completions.create(

        model=MODEL_NAME,

        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],

        temperature=0.2,

        max_tokens=700

    )

    return response.choices[0].message.content

In [6]:
answer = ask_llm(
    "What is an FIR?"
)

print(answer)

FIR stands for First Information Report. It is a document filed by a police officer in India when they receive information about a cognizable offense, which is a crime that can be investigated without a warrant. The FIR is the first step in the criminal investigation process and serves as a formal record of the incident.

The FIR typically includes the following details:

1. Date, time, and place of the incident
2. Names and contact information of the complainant (the person who reported the crime)
3. Names and contact information of the accused (the person or people suspected of committing the crime)
4. A brief description of the incident, including the nature of the offense and any injuries or property damage
5. The signature of the police officer who received the information and filed the FIR

The FIR is a crucial document in the Indian legal system, as it serves as the basis for further investigation and potential prosecution of the accused. It is usually filed at the nearest polic

In [7]:
# ============================================================
# Semantic Search using FAISS
# ============================================================

def retrieve_documents(query, top_k=3):

    # Generate query embedding
    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    )

    # Search FAISS
    scores, indices = index.search(
        query_embedding.astype("float32"),
        top_k
    )

    retrieved_docs = []

    for idx in indices[0]:

        if idx != -1:

            retrieved_docs.append(documents[idx])

    return retrieved_docs

In [9]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "BAAI/bge-small-en-v1.5"
)

print("✅ Embedding Model Loaded")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

✅ Embedding Model Loaded


In [11]:
print(embedding_model)

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'cls', 'include_prompt': True})
  (2): Normalize({})
)


In [13]:
import faiss
import os

BASE_PATH = "/content/drive/MyDrive/database"

faiss_path = os.path.join(BASE_PATH, "faiss.index")

index = faiss.read_index(faiss_path)

print("FAISS Loaded Successfully")
print("Total Indexed Documents:", index.ntotal)

FAISS Loaded Successfully
Total Indexed Documents: 50


In [14]:
import pickle

documents_path = os.path.join(BASE_PATH, "documents.pkl")

with open(documents_path, "rb") as f:
    documents = pickle.load(f)

print("Documents Loaded:", len(documents))

Documents Loaded: 50


In [15]:
print(type(index))
print(type(documents))
print(type(embedding_model))

<class 'faiss.swigfaiss.IndexFlatIP'>
<class 'list'>
<class 'sentence_transformers.sentence_transformer.model.SentenceTransformer'>


In [16]:
print(index.ntotal)

50


In [17]:
query = "Vehicle theft in Chennai"

results = retrieve_documents(query)

print("Retrieved Documents:", len(results))

Retrieved Documents: 3


In [18]:
print(len(documents))

50


In [20]:
for i, doc in enumerate(results):

    print("="*80)

    print("Document", i+1)

    print(doc.metadata)

    print()

Document 1
{'fir_number': 'TNFIR20260041', 'case_number': 'CASE041', 'crime': 'Vehicle Theft', 'district': 'Chennai', 'station': 'T. Nagar Police Station', 'criminal': 'Ashok', 'officer': 'R.Kumar 41', 'rank': 'Inspector', 'status': 'Under Investigation'}

Document 2
{'fir_number': 'TNFIR20260021', 'case_number': 'CASE021', 'crime': 'Vehicle Theft', 'district': 'Chennai', 'station': 'T. Nagar Police Station', 'criminal': 'Bala', 'officer': 'R.Kumar 21', 'rank': 'Inspector', 'status': 'Under Investigation'}

Document 3
{'fir_number': 'TNFIR20260016', 'case_number': 'CASE016', 'crime': 'Vehicle Theft', 'district': 'Chennai', 'station': 'T. Nagar Police Station', 'criminal': 'Lakshmi', 'officer': 'K.Murugan 16', 'rank': 'Inspector', 'status': 'Under Investigation'}



In [22]:
# ============================================================
# Build Context
# ============================================================

def build_context(documents):

    context = ""

    for i, doc in enumerate(documents):

        context += f"\n\n========== DOCUMENT {i+1} ==========\n"

        context += doc.page_content

    return context

In [23]:
context = build_context(results)

print(context[:2500])



========== DOCUMENT 1 ==========


================ FIR DETAILS ================

FIR Number : TNFIR20260041

Case Number : CASE041

Crime Category : Theft

Crime Type : Vehicle Theft

IPC Sections : 379

Description :

Vehicle Theft reported at Chennai

Investigation Status :

Under Investigation

Charge Sheet :

Pending

Police Station :

T. Nagar Police Station

District :

Chennai

Incident Date :

2026-07-14

Incident Time :

09:45


================ PERSON DETAILS ================

Person Name :

Ashok

Gender :

Male

DOB :

1991-06-14

Occupation :

Teacher

Address :

No.41, Gandhi Street

Mobile :

9800000041


================ CRIMINAL DETAILS ================

Alias :

Alias-41

Gang :

Red Tigers

Risk Level :

High

Previous Cases :

5

Arrest Count :

2

Repeat Offender :

No

Bail Status :

Denied

Current Status :

Under Investigation

Wanted :

No


================ OFFICER DETAILS ================

Officer Name :

R.Kumar 41

Rank :

Inspector

Station :

T. Nagar 

In [24]:
# ============================================================
# Build Prompt
# ============================================================

def build_prompt(question, context):

    prompt = f"""

You are an intelligent AI assistant for the Tamil Nadu Police Crime Database.

Your job is to answer ONLY using the information provided in the context.

Rules:

1. Do NOT make up information.

2. If the answer is not available in the context, reply:

"I could not find that information in the police database."

3. Give concise but complete answers.

4. Mention FIR numbers whenever possible.

5. Mention officer names, criminal names, police stations, investigation status, wanted status, warrants and court details whenever relevant.

--------------------------------------------------------

Context

{context}

--------------------------------------------------------

Question

{question}

--------------------------------------------------------

Answer

"""

    return prompt

In [25]:
question = "Who investigated FIR TNFIR20260001?"

prompt = build_prompt(question, context)

print(prompt[:3000])



You are an intelligent AI assistant for the Tamil Nadu Police Crime Database.

Your job is to answer ONLY using the information provided in the context.

Rules:

1. Do NOT make up information.

2. If the answer is not available in the context, reply:

"I could not find that information in the police database."

3. Give concise but complete answers.

4. Mention FIR numbers whenever possible.

5. Mention officer names, criminal names, police stations, investigation status, wanted status, warrants and court details whenever relevant.

--------------------------------------------------------

Context



========== DOCUMENT 1 ==========


================ FIR DETAILS ================

FIR Number : TNFIR20260041

Case Number : CASE041

Crime Category : Theft

Crime Type : Vehicle Theft

IPC Sections : 379

Description :

Vehicle Theft reported at Chennai

Investigation Status :

Under Investigation

Charge Sheet :

Pending

Police Station :

T. Nagar Police Station

District :

Chennai

In

In [26]:
# ============================================================
# Complete RAG Pipeline
# ============================================================

def rag_chat(question):

    # Retrieve relevant documents
    retrieved_docs = retrieve_documents(question)

    # Build context
    context = build_context(retrieved_docs)

    # Build prompt
    prompt = build_prompt(question, context)

    # Ask Groq
    answer = ask_llm(prompt)

    return answer

In [27]:
question = "Who investigated FIR TNFIR20260001?"

answer = rag_chat(question)

print(answer)

I could not find that information in the police database.


In [29]:
questions = [

    "Show all wanted criminals",

    "Who is Arun Kumar?",



]

for q in questions:

    print("="*100)

    print("Question:")

    print(q)

    print()

    print("Answer:")

    print(rag_chat(q))

    print()

Question:
Show all wanted criminals

Answer:
Based on the provided documents, the following individuals are wanted:

1. Senthil (Alias-23, Delta Gang) - Reward: 95000, Reason: Robbery, Last Seen: 2026-07-15, Location: Madurai, Issued By: O023, Active: Yes

2. Raghu (Alias-47, Delta Gang) - Reward: 85000, Reason: Chain snatching, Last Seen: 2026-09-02, Location: Tiruppur, Issued By: O047, Active: No

Question:
Who is Arun Kumar?

Answer:
Arun Kumar is a person whose details are available in the police database. He is a male, born on 1991-02-02, and is a teacher by occupation. His address is No.1, Gandhi Street, and his mobile number is 9800000001.



In [30]:
questions = [

    "Show vehicle theft cases",

    "Which FIR is under investigation?",

]

for q in questions:

    print("="*100)

    print("Question:")

    print(q)

    print()

    print("Answer:")

    print(rag_chat(q))

    print()

Question:
Show vehicle theft cases

Answer:
Based on the provided documents, the following vehicle theft cases are found:

1. FIR Number: TNFIR20260006, Case Number: CASE006, Incident Date: 2026-07-07
2. FIR Number: TNFIR20260041, Case Number: CASE041, Incident Date: 2026-07-14
3. FIR Number: TNFIR20260026, Case Number: CASE026, Incident Date: 2026-07-27

Question:
Which FIR is under investigation?

Answer:
The FIR numbers under investigation are TNFIR20260050, TNFIR20260005, and TNFIR20260035.



In [32]:
questions = [

    "Who is the arresting officer?",

    "What evidence is available for FIR TNFIR20260050?"

]

for q in questions:

    print("="*100)

    print("Question:")

    print(q)

    print()

    print("Answer:")

    print(rag_chat(q))

    print()

Question:
Who is the arresting officer?

Answer:
The arresting officer for the case TNFIR20260050 is O050.

Question:
What evidence is available for FIR TNFIR20260050?

Answer:
I could not find that information in the police database.

